# 7. API Keys and Environment Variables

Some APIs are public. Other APIs require an **API key**.

An API key is a secret string that identifies your app or account. API keys are commonly used for:

- access control
- billing or usage tracking
- rate limits
- abuse prevention


## The Most Important Rule

Do not paste a real API key into a notebook, GitHub repo, slide deck, or screenshot.

A common pattern is to store secrets in a local `.env` file and read them into Python as environment variables.


In [1]:
import os
from pathlib import Path
import requests

TIMEOUT = 10

## What a `.env` File Looks Like

A `.env` file is a plain text file with one `NAME=value` pair per line.

For example:

```text
OPENAI_API_KEY=sk-your-real-key-goes-here
WEATHER_API_KEY=your-weather-key-goes-here
```

The `.env` file should stay on your computer. It should not be committed to git.


In [2]:
example_env_text = """OPENAI_API_KEY=sk-your-real-key-goes-here
WEATHER_API_KEY=your-weather-key-goes-here
"""

print(example_env_text)

OPENAI_API_KEY=sk-your-real-key-goes-here
WEATHER_API_KEY=your-weather-key-goes-here



## Check Whether `.env` Is Ignored

This repo includes `.env` in `.gitignore`, which helps prevent accidental commits of real keys.


In [3]:
gitignore_path = Path(".gitignore")

if gitignore_path.exists():
    gitignore_text = gitignore_path.read_text()
    print(".gitignore exists")
    print(".env is ignored:", ".env" in gitignore_text)
else:
    print("No .gitignore file found yet.")

No .gitignore file found yet.


## Load Environment Variables

The `python-dotenv` package can read a local `.env` file and load its values into `os.environ`.

If the package is not installed, run this in a terminal:

```bash
python3 -m pip install python-dotenv
```


In [4]:
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None
    print("python-dotenv is not installed yet.")
else:
    loaded = load_dotenv()
    print("Tried to load .env file:", loaded)

Tried to load .env file: False


## Read an API Key Safely

This code checks whether `OPENAI_API_KEY` exists. It never prints the full key.


In [5]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    masked_key = api_key[:7] + "..." + api_key[-4:]
    print("OPENAI_API_KEY found:", masked_key)
else:
    print("No OPENAI_API_KEY found. That is okay for this lesson.")

No OPENAI_API_KEY found. That is okay for this lesson.


## Simulate an API Key Header

This uses a fake key and a real public testing API. The goal is to see where an API key usually goes in a request.


In [6]:
fake_api_key = "demo_not_a_real_key"
headers = {
    "Authorization": f"Bearer {fake_api_key}",
}

response = requests.get(
    "https://httpbin.org/headers",
    headers=headers,
    timeout=TIMEOUT,
)

print(response.status_code)
print(response.json()["headers"].get("Authorization"))

200
Bearer demo_not_a_real_key


## A Reusable Helper

A helper function can keep secret-handling code in one place.


In [7]:
def get_required_env_var(name):
    value = os.getenv(name)

    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")

    return value

try:
    key = get_required_env_var("OPENAI_API_KEY")
except RuntimeError as error:
    print(error)
else:
    print("Key loaded without printing it.")

Missing required environment variable: OPENAI_API_KEY


## Quick Review

- API keys are secrets.
- `.env` files are for local secrets.
- `.gitignore` helps keep `.env` out of GitHub.
- Python reads environment variables with `os.getenv()`.
- Authenticated APIs often receive keys in request headers.
